# AgentCore Payments — Media Content Monetization Demo

This notebook walks through the complete demo: exploring merchants, checking trust,
inspecting payment resources, and running the research agent.

**Prerequisites:** Run `demo/0-check-health.sh` first to verify all services are up.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from core import load_config, create_agent, DemoConfig
from core.callbacks import CollectorCallbacks, PrintCallbacks

config = load_config(env_file='../agent/.env')
print(f'Manager: {config.payment_manager_arn.split("/")[-1]}')
print(f'Instrument: {config.payment_instrument_id}')
print(f'Session: {config.payment_session_id}')
print(f'Merchant: {config.merchant_url}')

## Step 1: Explore Merchants

Discover what content is available across the 4 publishers.

In [ ]:
import httpx, json

merchants = httpx.get(f'{config.merchant_url}/merchants.json').json()
for mid, m in merchants['merchants'].items():
    print(f"  {m['name']:<20} {'✓ verified' if m['verified'] else '✗ unverified':<14} Standard: ${m['pricing']['standard']}")

In [ ]:
# Fetch catalogs
for merchant in ['mediatech', 'copperview', 'thornwick', 'kettlebrook']:
    try:
        catalog = httpx.get(f'{config.merchant_url}/{merchant}/catalog.json').json()
        print(f'\n--- {merchant} ({len(catalog.get("articles", []))} articles) ---')
        for a in catalog.get('articles', [])[:3]:
            print(f"  ${a.get('price', '?'):<8} {a.get('title', 'untitled')}")
    except Exception as e:
        print(f'  {merchant}: {e}')

## Step 2: Hit the Paywall (x402)

Request premium content without payment — see the 402 response.

In [ ]:
resp = httpx.get(f'{config.merchant_url}/mediatech/premium/agent-traffic-report.json')
print(f'HTTP Status: {resp.status_code}')
print()
payload = resp.json()
print(json.dumps(payload, indent=2))

## Step 3: Explore Trust Registry

Query merchant reputation — this is what the agent checks before spending money.

In [ ]:
for mid in ['mediatech-daily', 'copperview', 'thornwick', 'kettlebrook']:
    rep = httpx.get(f'{config.trust_registry_url}/merchants/{mid}/reputation').json()
    score = f"{rep['trustScore']}/5" if rep['trustScore'] else 'N/A'
    print(f"  {rep['name']:<20} Trust: {score:<8} Txns: {rep['totalTransactions']:<5} Disputes: {rep['disputeRate']*100:.0f}%")

## Step 4: Inspect Payment Resources

Check wallet balance and session budget.

In [ ]:
from bedrock_agentcore.payments.manager import PaymentManager

pm = PaymentManager(payment_manager_arn=config.payment_manager_arn, region_name=config.region)

# Instrument
inst = pm.get_payment_instrument(
    payment_instrument_id=config.payment_instrument_id,
    user_id=config.user_id)
wallet = inst.get('paymentInstrumentDetails', {}).get('embeddedCryptoWallet', {}).get('walletAddress', '?')
print(f'Wallet: {wallet}')
print(f'Status: {inst["status"]}')

# Balance
bal = pm.get_payment_instrument_balance(
    payment_connector_id=config.payment_connector_id,
    payment_instrument_id=config.payment_instrument_id,
    chain='BASE_SEPOLIA', token='USDC', user_id=config.user_id)
print(f'Balance: {bal["tokenBalance"]["amount"]} USDC')

# Session
sess = pm.get_payment_session(
    payment_session_id=config.payment_session_id,
    user_id=config.user_id)
avail = sess.get('availableLimits', {}).get('availableSpendAmount', {}).get('value', '?')
total = sess.get('limits', {}).get('maxSpendAmount', {}).get('value', '?')
print(f'Session: ${avail} available of ${total} budget')

## Step 5: Run the Research Agent

The agent discovers content, evaluates trust/quality/cost, makes purchase decisions,
and synthesizes findings. Watch the decision framework in action.

In [ ]:
agent = create_agent(config)

topic = config.research_topic
print(f'Topic: {topic}')
print(f'Budget: $1.00 USDC')
print('=' * 70)
print()

result = agent(f'Research the following topic: {topic}')
print(result)

## Step 6: Post-Run — Check Remaining Budget

In [ ]:
sess_after = pm.get_payment_session(
    payment_session_id=config.payment_session_id,
    user_id=config.user_id)
avail_after = sess_after.get('availableLimits', {}).get('availableSpendAmount', {}).get('value', '?')
print(f'Budget remaining: ${avail_after} of ${total}')
print(f'Spent: ${float(total) - float(avail_after):.4f}')

## Step 7: Verify On-Chain

Open the block explorer to see actual USDC transfers.

In [ ]:
from IPython.display import Markdown
Markdown(f'[View wallet transactions on Base Sepolia](https://sepolia.basescan.org/address/{wallet})')